In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from shapely.geometry import Point
import geopandas
import matplotlib.pyplot as plt
import json

In [2]:
hawaii = 'https://scholarspace.manoa.hawaii.edu/communities/533eb99e-2e59-47a5-9c4e-652352bfc5a6'

In [3]:
hawaii = requests.get(hawaii).text

In [4]:
hawaii_soup = BeautifulSoup(hawaii, 'html.parser')

In [5]:
def extract_hrefs(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    hrefs = [a['href'] for a in soup.find_all('a', class_='lead ng-star-inserted', href=True)]
    return hrefs

In [12]:
base_url = 'https://scholarspace.manoa.hawaii.edu/communities/533eb99e-2e59-47a5-9c4e-652352bfc5a6?cmcl.page='

# List to store all hrefs
all_hrefs = []

# Iterate over pages
for page_number in range(1, 10):  # Adjust the range as needed
    page_url = f'{base_url}{page_number}'
    hrefs = extract_hrefs(page_url)
    all_hrefs.extend(hrefs)

# Print all hrefs
for href in all_hrefs:
    print(href)

/collections/8e4c3ded-2f0b-44ad-98ff-6d93969f029e
/collections/bf4ed241-69d9-477a-81a3-1605d499316f
/collections/d49da7ee-c81f-4fed-acdb-efa9f0ed5970
/collections/7b05d7fa-597b-4a2c-918e-f68d29a91d86
/collections/4a1b41b0-8e05-48cd-b1cc-d198198b976c
/collections/1c44423e-902c-4e26-8c44-761cd017fa41
/collections/65d98780-f7ba-4523-8b37-70a37da197f2
/collections/686e028b-a6b0-44ab-a5a1-6757baad7ac5
/collections/4901cefe-4918-4ee9-8db6-06dff65280b0
/collections/3b819417-6911-4df2-990d-6bab4507448c
/collections/9433b5ba-f912-4275-aa10-e076b4f3f743
/collections/21c604e1-6719-4869-84e5-95cc6cb28c91
/collections/823bd02a-9436-402f-95f3-1f89315a30b0
/collections/82c0d02b-5c31-43e6-a581-66678f94b544
/collections/a1426c5f-3e3a-403d-893b-7b6ce96e5d1b
/collections/9d0c315c-ca16-4a59-94eb-b66c61e605e4
/collections/13d56c46-9bf4-4877-bdaf-bdcb44b32496
/collections/e1cf3b4d-ba28-4d2b-bad0-a49356b9060f
/collections/e23bb2a5-4764-4230-9681-3eafe639bf9e
/collections/df6553f8-1c51-42bc-b92c-d283d3f3e694


In [13]:
prefix = 'https://scholarspace.manoa.hawaii.edu'
for i in range(len(all_hrefs)):
    all_hrefs[i] = prefix + all_hrefs[i]

In [14]:
all_hrefs

['https://scholarspace.manoa.hawaii.edu/collections/8e4c3ded-2f0b-44ad-98ff-6d93969f029e',
 'https://scholarspace.manoa.hawaii.edu/collections/bf4ed241-69d9-477a-81a3-1605d499316f',
 'https://scholarspace.manoa.hawaii.edu/collections/d49da7ee-c81f-4fed-acdb-efa9f0ed5970',
 'https://scholarspace.manoa.hawaii.edu/collections/7b05d7fa-597b-4a2c-918e-f68d29a91d86',
 'https://scholarspace.manoa.hawaii.edu/collections/4a1b41b0-8e05-48cd-b1cc-d198198b976c',
 'https://scholarspace.manoa.hawaii.edu/collections/1c44423e-902c-4e26-8c44-761cd017fa41',
 'https://scholarspace.manoa.hawaii.edu/collections/65d98780-f7ba-4523-8b37-70a37da197f2',
 'https://scholarspace.manoa.hawaii.edu/collections/686e028b-a6b0-44ab-a5a1-6757baad7ac5',
 'https://scholarspace.manoa.hawaii.edu/collections/4901cefe-4918-4ee9-8db6-06dff65280b0',
 'https://scholarspace.manoa.hawaii.edu/collections/3b819417-6911-4df2-990d-6bab4507448c',
 'https://scholarspace.manoa.hawaii.edu/collections/9433b5ba-f912-4275-aa10-e076b4f3f743',

In [16]:
tr = all_hrefs[0:10]

In [17]:
tr

['https://scholarspace.manoa.hawaii.edu/collections/8e4c3ded-2f0b-44ad-98ff-6d93969f029e',
 'https://scholarspace.manoa.hawaii.edu/collections/bf4ed241-69d9-477a-81a3-1605d499316f',
 'https://scholarspace.manoa.hawaii.edu/collections/d49da7ee-c81f-4fed-acdb-efa9f0ed5970',
 'https://scholarspace.manoa.hawaii.edu/collections/7b05d7fa-597b-4a2c-918e-f68d29a91d86',
 'https://scholarspace.manoa.hawaii.edu/collections/4a1b41b0-8e05-48cd-b1cc-d198198b976c',
 'https://scholarspace.manoa.hawaii.edu/collections/1c44423e-902c-4e26-8c44-761cd017fa41',
 'https://scholarspace.manoa.hawaii.edu/collections/65d98780-f7ba-4523-8b37-70a37da197f2',
 'https://scholarspace.manoa.hawaii.edu/collections/686e028b-a6b0-44ab-a5a1-6757baad7ac5',
 'https://scholarspace.manoa.hawaii.edu/collections/4901cefe-4918-4ee9-8db6-06dff65280b0',
 'https://scholarspace.manoa.hawaii.edu/collections/3b819417-6911-4df2-990d-6bab4507448c']

In [30]:
def extract_hrefs(url, max_pages=30):
    all_hrefs = []
    visited_hrefs = set()
    page_num = 1
    while page_num <= max_pages:
        print(page_num)
        response = requests.get(url + f'?cp.page={page_num}')
        # Check if the response status code indicates success (200)
        if response.status_code != 200:
            break
        soup = BeautifulSoup(response.text, 'html.parser')
        links = soup.find_all('a', {'class': 'lead item-list-title dont-break-out ng-star-inserted'})
        inner_hrefs = [link['href'] for link in links]
        if not inner_hrefs:
            break
        for href in inner_hrefs:
            if href in visited_hrefs:
                # If we encounter a href that we've already visited, break the loop
                break
            all_hrefs.append(href)
            visited_hrefs.add(href)
        else:
            page_num += 1
            continue
        break  # Break the while loop if we encounter a visited href
    return all_hrefs


# Iterate over each webpage URL and extract href attributes
inner_hrefs = []
for url in tr:
    hrefs = extract_hrefs(url)
    inner_hrefs.extend(hrefs)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
1
2
1
2
3
4
5
6
7
8
9
10
1
2
3
4
5
1
2
3
4
5
6
1
2
3
4
5
6
7
8
9
10
11
12
1
2
3
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
1
2
1
2


In [57]:
for i in range(len(inner_hrefs)):
    inner_hrefs[i] = prefix + inner_hrefs[i] + '/full'

In [58]:
inner_hrefs

['https://scholarspace.manoa.hawaii.edu/items/53e7861f-e0f0-4d26-90f8-3f8ef82760a1/full',
 'https://scholarspace.manoa.hawaii.edu/items/82cafeb5-2441-4edf-9d3f-1f8b1e723335/full',
 'https://scholarspace.manoa.hawaii.edu/items/e6b72c2a-f4ca-4cd2-990f-a81eaec7b17a/full',
 'https://scholarspace.manoa.hawaii.edu/items/0d6233cb-aa0d-4bc9-b53f-374f6c94989b/full',
 'https://scholarspace.manoa.hawaii.edu/items/b307cf12-62a4-4c4c-965f-e3f2bccf125d/full',
 'https://scholarspace.manoa.hawaii.edu/items/dbcfa04a-384f-48db-86ca-0ad39abe33ed/full',
 'https://scholarspace.manoa.hawaii.edu/items/2975aeac-a453-43f4-b3da-19c86ff10540/full',
 'https://scholarspace.manoa.hawaii.edu/items/86cb7109-2f27-45b3-a3cb-679d9c26fdbb/full',
 'https://scholarspace.manoa.hawaii.edu/items/efdefd1b-d3a2-41ce-a8b2-88215c95ade5/full',
 'https://scholarspace.manoa.hawaii.edu/items/de42dc32-8caf-4aca-a194-b5402acfaf2e/full',
 'https://scholarspace.manoa.hawaii.edu/items/1c14c4f0-f8c5-4dc3-a06c-bdf5fb59a3e1/full',
 'https://

In [61]:
small_inner_hrefs = inner_hrefs[0:10]

In [68]:
# Function to scrape content language and code from inner_href
def scrape_content_language(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    language_tag = soup.find('td', text='dc.content.language')
    language_code_tag = soup.find('td', text='dc.content.languagecode')
    if language_tag and language_code_tag:
        language = language_tag.find_next_sibling('td').text.strip()
        language_code = language_code_tag.find_next_sibling('td').text.strip()
        return language, language_code
    else:
        return None, None

# Function to scrape file name and size
def scrape_files_info(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    file_sections = soup.find_all('div', class_='file-section row mb-3 ng-star-inserted')
    file_names = []
    file_sizes = []
    for section in file_sections:
        name_tag = section.find('dt', text='Name:')
        size_tag = section.find('dt', text='Size:')
        if name_tag and size_tag:
            file_names.append(name_tag.find_next_sibling('dd').text.strip())
            file_sizes.append(size_tag.find_next_sibling('dd').text.strip())
    return file_names, file_sizes



In [87]:
# Initialize lists to store scraped data
languages = []
language_codes = []
file_names_list = []
file_sizes_list = []

# Scrape data for each inner href
for href in small_inner_hrefs:
    language, language_code = scrape_content_language(href)
    if language and language_code:
        languages.append(language)
        language_codes.append(language_code)
        file_names, file_sizes = scrape_files_info(href)
        file_names_list.append(file_names)
        file_sizes_list.append(file_sizes)
        origin_href = href

# Create DataFrame
data = {
    'Href': origin_href,
    'Content Language': languages,
    'Language Code': language_codes,
    'File Name': file_names_list ,
    'File Size': file_sizes_list 
}

results = pd.DataFrame(data)


/var/folders/hw/3z11r39j0gd4kgzxbjj6v8z80000gp/T/ipykernel_9273/3010703008.py:5: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  language_tag = soup.find('td', text='dc.content.language')
/var/folders/hw/3z11r39j0gd4kgzxbjj6v8z80000gp/T/ipykernel_9273/3010703008.py:6: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  language_code_tag = soup.find('td', text='dc.content.languagecode')
/var/folders/hw/3z11r39j0gd4kgzxbjj6v8z80000gp/T/ipykernel_9273/3010703008.py:22: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  name_tag = section.find('dt', text='Name:')
/var/folders/hw/3z11r39j0gd4kgzxbjj6v8z80000gp/T/ipykernel_9273/3010703008.py:23: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  size_tag = section.find('dt', text='Size:')


In [88]:
results

,Href,Content Language,Language Code,File Name,File Size
0,https://scholarspace.manoa.hawaii.edu/items/de...,Palu'e,ple,"[SD1-335.eaf, SD1-335.wav]","[56.57 KB, 40.7 MB]"
1,https://scholarspace.manoa.hawaii.edu/items/de...,Palu'e,ple,"[SD1-334.eaf, SD1-334.wav]","[74.04 KB, 52.38 MB]"
2,https://scholarspace.manoa.hawaii.edu/items/de...,Palu'e,ple,"[SD1-333.eaf, SD1-333.wav]","[18.49 KB, 13.72 MB]"
3,https://scholarspace.manoa.hawaii.edu/items/de...,Palu'e,ple,"[SD1-332.eaf, SD1-332.wav]","[56.84 KB, 65.88 MB]"
4,https://scholarspace.manoa.hawaii.edu/items/de...,Palu'e,ple,"[SD1-331.eaf, SD1-331.mp4, SD1-331.wav]","[303.22 KB, 428.28 MB, 432.54 MB]"
5,https://scholarspace.manoa.hawaii.edu/items/de...,Palu'e,ple,"[SD1-330.mp4, SD1-330.wav, SD1-330.eaf]","[227.32 MB, 320.78 MB, 376.56 KB]"
6,https://scholarspace.manoa.hawaii.edu/items/de...,Palu'e,ple,"[SD1-329.mp4, SD1-329.wav, SD1-329.eaf]","[183.93 MB, 295.94 MB, 247.03 KB]"
7,https://scholarspace.manoa.hawaii.edu/items/de...,Palu'e,ple,"[SD1-328.wav, SD1-328.eaf]","[54.01 MB, 64.71 KB]"
8,https://scholarspace.manoa.hawaii.edu/items/de...,Palu'e,ple,"[SD1-327.wav, SD1-327.eaf]","[7.39 MB, 16.44 KB]"
9,https://scholarspace.manoa.hawaii.edu/items/de...,Palu'e,ple,"[SD1-326.wav, SD1-326.eaf]","[60.49 MB, 53.48 KB]"


In [81]:
len(data['Content Language'])

10

In [84]:
data['Content Language']

["Palu'e",
 "Palu'e",
 "Palu'e",
 "Palu'e",
 "Palu'e",
 "Palu'e",
 "Palu'e",
 "Palu'e",
 "Palu'e",
 "Palu'e"]

In [83]:
len(data['File Size'])

10